In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по свиньям v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Свиньи
1707,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2017-09-01,1031.57
123,АКМОЛИНСКАЯ ОБЛАСТЬ,2025-04-01,467.44
127,АКТЮБИНСКАЯ ОБЛАСТЬ,2015-01-01,779.40
148,АКТЮБИНСКАЯ ОБЛАСТЬ,2016-10-01,962.96
1725,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2019-03-01,1062.78
848,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2024-07-01,19.00
1372,МАНГИСТАУСКАЯ ОБЛАСТЬ,2015-04-01,1.44
451,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2017-03-01,721.24
1537,ОБЛАСТЬ ҰЛЫТАУ,2024-05-01,1.10
1173,КОСТАНАЙСКАЯ ОБЛАСТЬ,2019-11-01,3072.44


In [3]:
# === загружаем данные ===
best_methods = pd.read_excel("results/Свиньи - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АЛМАТИНСКАЯ ОБЛАСТЬ,193.80,71.08,196.48,67.11,302.86,89.04,HW,MAPE,193.80,67.11
1,ОБЛАСТЬ АБАЙ,34.83,28.26,38.66,30.21,75.97,87.35,HW,MAPE,34.83,28.26
2,ОБЛАСТЬ ЖЕТІСУ,58.23,117.75,168.03,296.42,226.78,503.12,HW,MAPE,58.23,117.75
3,АКМОЛИНСКАЯ ОБЛАСТЬ,35.99,167.22,34.23,157.56,31.66,147.15,Prophet,MAPE,31.66,147.15
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,27.16,89.04,34.98,119.29,19.05,68.49,Prophet,MAPE,19.05,68.49
5,КАРАГАНДИНСКАЯ ОБЛАСТЬ,15.53,150.24,16.57,154.15,15.19,152.82,Prophet,MAPE,15.19,150.24
6,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,NaN,1.59,181.89,1.61,73.34,1.25,Prophet,MAPE,73.34,1.25
7,АКТЮБИНСКАЯ ОБЛАСТЬ,23.21,11.83,17.36,9.25,59.60,42.91,SARIMA,MAPE,17.36,9.25
8,ГШЫМКЕНТ,96.04,2.55,38.45,0.62,58.29,3.21,SARIMA,MAPE,38.45,0.62
9,ЖАМБЫЛСКАЯ ОБЛАСТЬ,41.37,9.95,24.22,6.87,40.19,13.11,SARIMA,MAPE,24.22,6.87


In [4]:
actual_aug = pd.read_excel("Свиньи 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["Свиньи"] = (actual_aug["Свиньи"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("Свиньи обработанные август 2025.xlsx", index=False)
actual_aug



,Регион,Период,Свиньи
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2025-08-01,547.75
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2025-08-01,7.70
2,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-08-01,182.50
3,АТЫРАУСКАЯ ОБЛАСТЬ,2025-08-01,NaN
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-08-01,149.20
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2025-08-01,13.00
6,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2025-08-01,866.34
7,КОСТАНАЙСКАЯ ОБЛАСТЬ,2025-08-01,349.15
8,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2025-08-01,4.60
9,МАНГИСТАУСКАЯ ОБЛАСТЬ,2025-08-01,NaN


In [5]:
# === настройки ===
TARGET = "Свиньи"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [6]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [7]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [8]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [9]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [10]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [11]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/Свиньи - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug

19:15:18 - cmdstanpy - INFO - Chain [1] start processing
19:15:18 - cmdstanpy - INFO - Chain [1] done processing
19:15:19 - cmdstanpy - INFO - Chain [1] start processing
19:15:19 - cmdstanpy - INFO - Chain [1] done processing
19:15:19 - cmdstanpy - INFO - Chain [1] start processing
19:15:19 - cmdstanpy - INFO - Chain [1] done processing
19:15:19 - cmdstanpy - INFO - Chain [1] start processing
19:15:19 - cmdstanpy - INFO - Chain [1] done processing


,Регион,Лучший метод,Прогноз (2025-08),Факт (2025-08),"Отклонение, %"
0,АКМОЛИНСКАЯ ОБЛАСТЬ,Prophet,337.31,547.75,-38.42
1,АКТЮБИНСКАЯ ОБЛАСТЬ,SARIMA,7.37,7.70,-4.31
2,АЛМАТИНСКАЯ ОБЛАСТЬ,HW,87.08,182.50,-52.28
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,Prophet,268.01,548.68,-51.15
4,ГШЫМКЕНТ,SARIMA,0.59,NaN,NaN
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,SARIMA,7.86,13.00,-39.56
6,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,SARIMA,173.73,149.20,16.44
7,КАРАГАНДИНСКАЯ ОБЛАСТЬ,Prophet,810.33,866.34,-6.47
8,КОСТАНАЙСКАЯ ОБЛАСТЬ,SARIMA,441.36,349.15,26.41
9,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,SARIMA,3.69,4.60,-19.86
